# 🎯 PlacePulse AI — B.Tech Placement Probability Classifier
### Model 1: Binary Placement Prediction Pipeline (Zero-Leakage & 5-Fold CV)
**Target Audience:** B.Tech Engineering Students (CSE, IT, ECE, Mech, Civil, EE)  
**Kaggle Dataset:** `tejashvi14/engineering-placements-prediction` (`collegePlace.csv` - 2,966 records)  
**Direct Kaggle Link:** https://www.kaggle.com/datasets/tejashvi14/engineering-placements-prediction  
**Algorithms Benchmarked:** Dummy Baseline (Most Frequent), Logistic Regression (L2), Random Forest (100 trees), XGBoost, Decision Tree, Gradient Boosting

---
### Data Science Integrity Guarantees:
1. **Zero Data Leakage:** Preprocessing (`StandardScaler`, `OneHotEncoder`) is encapsulated strictly inside Scikit-Learn `Pipeline` and fit only on training folds.
2. **Duplicate Audit:** Checks and drops duplicate records before splitting to prevent synthetic metric inflation.
3. **Model Selection:** Winning estimator selected based on **mean 5-fold cross-validation F1-score on training split**.
4. **Unbiased Holdout Evaluation:** Test set is isolated upfront and evaluated strictly **ONCE**.
5. **Masked Authentication:** Kaggle API key entered securely using `getpass`.


### 📦 Step 1: Install Required Libraries


In [ ]:
!pip install -q kaggle scikit-learn xgboost pandas numpy joblib matplotlib seaborn
print("Dependencies installed!")


### 🔑 Step 2: Masked Kaggle Authentication (`getpass` or `kaggle.json`)
Upload your `kaggle.json` or enter credentials securely without echoing key:


In [ ]:
import os, json, getpass
from google.colab import files

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")

print("Upload kaggle.json OR leave empty to input securely:")
uploaded = files.upload()

if "kaggle.json" in uploaded:
    with open(kaggle_json_path, "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod(kaggle_json_path, 0o600)
    print("Successfully authenticated with uploaded kaggle.json!")
else:
    u = input("Kaggle Username: ").strip()
    k = getpass.getpass("Enter Kaggle API Key (hidden): ").strip()
    if u and k:
        with open(kaggle_json_path, "w") as f:
            json.dump({"username": u, "key": k}, f)
        os.chmod(kaggle_json_path, 0o600)
        print("Successfully authenticated with masked credentials!")


### 📥 Step 3: Resilient Dataset Ingestion & Duplicate Check


In [ ]:
import os, urllib.request
import pandas as pd
import numpy as np

os.makedirs("data/raw", exist_ok=True)
os.makedirs("models", exist_ok=True)

csv_path = "data/raw/collegePlace.csv"

# Tier 1: Try Kaggle CLI
if not os.path.exists(csv_path):
    print("Attempting download via Kaggle CLI (tejashvi14/engineering-placements-prediction)...")
    !kaggle datasets download -d tejashvi14/engineering-placements-prediction -p data/raw --unzip || true

# Tier 2: Public HTTPS Raw Mirror Fallback (bypasses Kaggle 403 Forbidden)
if not os.path.exists(csv_path):
    print("Kaggle CLI returned 403/Forbidden or file missing. Downloading from verified public mirror...")
    mirror_urls = [
        "https://raw.githubusercontent.com/Avi8010/Placement_Predictor/master/collegePlace.csv",
        "https://raw.githubusercontent.com/Nileshshinde09/student-placement-prediction-app-using-online-machine-learning/main/collegePlace.csv"
    ]
    for url in mirror_urls:
        try:
            print(f"Fetching from: {url}")
            urllib.request.urlretrieve(url, csv_path)
            if os.path.exists(csv_path) and os.path.getsize(csv_path) > 1000:
                print("✓ Successfully downloaded collegePlace.csv from public mirror!")
                break
        except Exception as e:
            print(f"Mirror failed ({e}), trying next...")

# Tier 3: Manual upload fallback
if not os.path.exists(csv_path):
    from google.colab import files
    print("Please upload collegePlace.csv manually:")
    up = files.upload()
    for fname in up:
        if "college" in fname.lower() and fname.endswith(".csv"):
            os.rename(fname, csv_path)
            break

df_raw = pd.read_csv(csv_path)
print(f"Total Raw Rows: {len(df_raw)}")

# Duplicate Check (Point 12)
dup_count = df_raw.duplicated().sum()
print(f"Duplicate records found: {dup_count}")
df = df_raw.drop_duplicates().copy()
print(f"Deduplicated records for training: {len(df)}")

# Stream category check (Small things)
print("Stream categories in dataset:", list(df["Stream"].unique()))
display(df.head(3))


### 🛠️ Step 4: Cleaner Pipeline Formulation & 5-Fold Cross Validation (Points 5 & 6)
We construct the non-linear interaction term $CGPA \times (Internships + 1)$ and assemble a `ColumnTransformer`.
Models are selected strictly via 5-Fold Stratified Cross-Validation on the training split, with preprocessing embedded inside each fold.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import joblib

# Feature Interaction
df["CGPA_x_Intern"] = df["CGPA"] * (df["Internships"] + 1)
STREAMS = ["Computer Science", "Information Technology", "Electronics And Communication", "Mechanical", "Electrical", "Civil"]

X = df[["CGPA", "CGPA_x_Intern", "Stream", "Internships", "HistoryOfBacklogs"]]
y = df["PlacedOrNot"].values

# 80/20 Holdout Split (Touched strictly ONCE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training split: {len(X_train)} rows | Holdout test split: {len(X_test)} rows")

# Preprocessor: fits strictly on training folds
pre = ColumnTransformer([
    ("num", StandardScaler(), ["CGPA", "CGPA_x_Intern"]),
    ("cat", OneHotEncoder(categories=[STREAMS], handle_unknown="ignore", sparse_output=False), ["Stream"]),
    ("keep", "passthrough", ["Internships", "HistoryOfBacklogs"])
])

models = {
    "Baseline (Most Frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression (L2)": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "Random Forest (100 trees)": RandomForestClassifier(n_estimators=100, max_depth=8, criterion="entropy", random_state=42),
    "XGBoost Classifier": xgb.XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, eval_metric="logloss", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
best_cv_f1 = -1.0
best_model_name = ""

print("-" * 75)
print(f"{'Classifier Model':<28} | {'CV Accuracy':<12} | {'CV F1-Score':<12} | {'CV ROC-AUC':<10}")
print("-" * 75)

for name, clf in models.items():
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["accuracy", "f1", "roc_auc"])
    mean_acc = float(np.mean(scores["test_accuracy"]))
    mean_f1 = float(np.mean(scores["test_f1"]))
    mean_auc = float(np.mean(scores["test_roc_auc"]))
    
    cv_results[name] = {"accuracy": mean_acc, "f1": mean_f1, "roc_auc": mean_auc}
    print(f"{name:<28} | {mean_acc:.4f}       | {mean_f1:.4f}       | {mean_auc:.4f}")
    
    if mean_f1 > best_cv_f1:
        best_cv_f1 = mean_f1
        best_model_name = name

print("-" * 75)
print(f"Selected Winner via 5-Fold Cross-Validation: '{best_model_name}' (Mean CV F1: {best_cv_f1:.4f})")


### 📊 Step 5: Final Evaluation on Holdout Test Split (Touched ONCE)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

winner_pipe = Pipeline([
    ("pre", pre),
    ("clf", models[best_model_name])
])
winner_pipe.fit(X_train, y_train)

y_pred = winner_pipe.predict(X_test)
y_prob = winner_pipe.predict_proba(X_test)[:, 1] if hasattr(winner_pipe, "predict_proba") else y_pred

test_acc = accuracy_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred, zero_division=0)
test_rec = recall_score(y_test, y_pred, zero_division=0)
test_f1 = f1_score(y_test, y_pred, zero_division=0)
test_auc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print(f"UNBIASED TEST EVALUATION FOR {best_model_name.upper()}:")
print("=" * 60)
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall:    {test_rec:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_auc:.4f}")

# Confusion matrix plot
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", xticklabels=["Unplaced", "Placed"], yticklabels=["Unplaced", "Placed"])
plt.title(f"{best_model_name} Test Confusion Matrix", fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


### 📦 Step 6: Export Production Pipeline & Auto-Download


In [ ]:
import zipfile
from google.colab import files

joblib.dump(winner_pipe, "models/placement_pipeline.joblib")
with open("models/classifier_benchmarks.json", "w") as f:
    json.dump({
        "production_model": best_model_name,
        "best_cv_f1": best_cv_f1,
        "test_metrics": {"accuracy": test_acc, "f1": test_f1, "roc_auc": test_auc},
        "cv_leaderboard": cv_results
    }, f, indent=2)

zip_name = "placepulse_placement_model.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("models/placement_pipeline.joblib", "placement_pipeline.joblib")
    z.write("models/classifier_benchmarks.json", "classifier_benchmarks.json")
    z.write("data/raw/collegePlace.csv", "data/raw/collegePlace.csv")

print(f"Archive ready: {zip_name}")
files.download(zip_name)
print("Pipeline downloaded! Place it in placepulse_ds_lab/models/.")
